In [ ]:
import torch
import pandas as pd
import numpy as np
import string
from torch import nn
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag

In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt_tab')

In [87]:
from torch import optim

In [ ]:
!unzip /content/archive.zip

In [88]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [89]:
train_data = pd.read_csv("/kaggle/input/sentiment-analysis-dataset/train.csv", encoding='latin1')

In [90]:
train_data

,textID,text,selected_text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (Km²),Density (P/Km²)
0,cb774db0d1,"I`d have responded, if I were going","I`d have responded, if I were going",neutral,morning,0-20,Afghanistan,38928346,652860.0,60
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative,noon,21-30,Albania,2877797,27400.0,105
2,088c60f138,my boss is bullying me...,bullying me,negative,night,31-45,Algeria,43851044,2381740.0,18
3,9642c003ef,what interview! leave me alone,leave me alone,negative,morning,46-60,Andorra,77265,470.0,164
4,358bd9e861,"Sons of ****, why couldn`t they put them on t...","Sons of ****,",negative,noon,60-70,Angola,32866272,1246700.0,26
...,...,...,...,...,...,...,...,...,...,...
27476,4eac33d1c0,wish we could come see u on Denver husband l...,d lost,negative,night,31-45,Ghana,31072940,227540.0,137
27477,4f4c4fc327,I`ve wondered about rake to. The client has ...,", don`t force",negative,morning,46-60,Greece,10423054,128900.0,81
27478,f67aae2310,Yay good for both of you. Enjoy the break - y...,Yay good for both of you.,positive,noon,60-70,Grenada,112523,340.0,331
27479,ed167662a5,But it was worth it ****.,But it was worth it ****.,positive,night,70-100,Guatemala,17915568,107160.0,167


In [91]:
train_data.columns

Index(['textID', 'text', 'selected_text', 'sentiment', 'Time of Tweet',
       'Age of User', 'Country', 'Population -2020', 'Land Area (Km²)',
       'Density (P/Km²)'],
      dtype='object')

In [92]:
y_train = train_data['sentiment']

In [93]:
y_train

0         neutral
1        negative
2        negative
3        negative
4        negative
           ...   
27476    negative
27477    negative
27478    positive
27479    positive
27480     neutral
Name: sentiment, Length: 27481, dtype: object

In [94]:
train_data.columns

Index(['textID', 'text', 'selected_text', 'sentiment', 'Time of Tweet',
       'Age of User', 'Country', 'Population -2020', 'Land Area (Km²)',
       'Density (P/Km²)'],
      dtype='object')

In [95]:
x_train = train_data.drop(columns = ['text', 'sentiment', 'Time of Tweet',
       'Age of User', 'Country', 'Population -2020', 'Land Area (Km²)',
       'Density (P/Km²)'])

In [96]:
x_train

,textID,selected_text
0,cb774db0d1,"I`d have responded, if I were going"
1,549e992a42,Sooo SAD
2,088c60f138,bullying me
3,9642c003ef,leave me alone
4,358bd9e861,"Sons of ****,"
...,...,...
27476,4eac33d1c0,d lost
27477,4f4c4fc327,", don`t force"
27478,f67aae2310,Yay good for both of you.
27479,ed167662a5,But it was worth it ****.


In [97]:
def word_count(sentence):
    return len(str(sentence).split())

temp = x_train['selected_text'].apply(word_count)

maxlen = temp.max()

In [98]:
maxlen

33

In [99]:
test_data = pd.read_csv("/kaggle/input/sentiment-analysis-dataset/test.csv", encoding='latin')

In [100]:
test_data

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,textID,text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (Km²),Density (P/Km²)
0,f87dea47db,Last session of the day http://twitpic.com/67ezh,neutral,morning,0-20,Afghanistan,38928346.0,652860.0,60.0
1,96d74cb729,Shanghai is also really exciting (precisely -...,positive,noon,21-30,Albania,2877797.0,27400.0,105.0
2,eee518ae67,"Recession hit Veronique Branquinho, she has to...",negative,night,31-45,Algeria,43851044.0,2381740.0,18.0
3,01082688c6,happy bday!,positive,morning,46-60,Andorra,77265.0,470.0,164.0
4,33987a8ee5,http://twitpic.com/4w75p - I like it!!,positive,noon,60-70,Angola,32866272.0,1246700.0,26.0
...,...,...,...,...,...,...,...,...,...
4810,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4811,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4812,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4813,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [101]:
y_test = test_data['sentiment']

In [102]:
y_test = y_test.dropna()

In [103]:
x_test = test_data.drop(columns = ['textID', 'sentiment','Time of Tweet', 'Age of User',
       'Country', 'Population -2020', 'Land Area (Km²)', 'Density (P/Km²)'])

In [104]:
x_test = x_test.dropna(axis = 0)

In [105]:
y_test

0        neutral
1       positive
2       negative
3       positive
4       positive
          ...   
3529    negative
3530    positive
3531    negative
3532    positive
3533    positive
Name: sentiment, Length: 3534, dtype: object

In [106]:
x_test

,text
0,Last session of the day http://twitpic.com/67ezh
1,Shanghai is also really exciting (precisely -...
2,"Recession hit Veronique Branquinho, she has to..."
3,happy bday!
4,http://twitpic.com/4w75p - I like it!!
...,...
3529,"its at 3 am, im very tired but i can`t sleep ..."
3530,All alone in this old house again. Thanks for...
3531,I know what you mean. My little dog is sinkin...
3532,_sutra what is your next youtube video gonna b...


In [ ]:
def preprocess_text(text):
    """
    Remove punctuation and stop words from text
    Returns: cleaned text as string
    """
    # Convert to lowercase
    text = str(text).lower()
    
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove stop words
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    
    # Join back to string
    return ' '.join(tokens)

In [ ]:
def add_pos_tags(text):
    """
    Add POS tags to text
    Returns: list of (word, POS_tag) tuples
    """
    tokens = word_tokenize(str(text).lower())
    
    # Remove punctuation from tokens
    tokens = [word for word in tokens if word not in string.punctuation]
    
    # POS tagging
    pos_tags = pos_tag(tokens)
    
    return pos_tags

In [ ]:
x_train['selected_text'] = x_train['selected_text'].apply(preprocess_text)

In [ ]:
x_test['text'] = x_test['text'].apply(preprocess_text)

In [107]:
from torch import nn

In [108]:
embed_dim = 33
#seq_len = maxlen
num_layers = 10

In [110]:
vocab_size = 33
hidden_dim = 10

In [109]:
# Run in a notebook cell first:
!pip install --upgrade transformers huggingface_hub

# Then restart kernel and run your code

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [111]:
from transformers import AutoTokenizer

# 1️⃣ Clean your text column
x_train['selected_text'] = x_train['selected_text'].astype(str)        # convert all entries to strings
x_train['selected_text'] = x_train['selected_text'].fillna('')         # replace NaN with empty string

# 2️⃣ Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")

# 3️⃣ Tokenize safely
encodings = tokenizer(
    list(x_train['selected_text']),   # guaranteed list of strings
    padding='max_length',
    truncation=True,
    max_length=33,
    return_tensors='pt'
)

print("input_ids shape:", encodings['input_ids'].shape)


input_ids shape: torch.Size([27481, 33])


In [112]:
encodings.get('input_ids')

tensor([[  101,  1045,  1036,  ...,     0,     0,     0],
        [  101, 17111,  2080,  ...,     0,     0,     0],
        [  101, 18917,  2033,  ...,     0,     0,     0],
        ...,
        [  101,  8038,  2100,  ...,     0,     0,     0],
        [  101,  2021,  2009,  ...,     0,     0,     0],
        [  101,  2035,  2023,  ...,     0,     0,     0]])

In [114]:
x_train = encodings.get('input_ids').to(device)

In [115]:
y_train = y_train.apply(lambda x: 0 if x == 'neutral' else (1 if x == 'positive' else 2))

In [116]:
y_test = y_test.apply(lambda x: 0 if x == 'neutral' else (1 if x == 'positive' else 2))

In [117]:
y_train = torch.tensor(y_train).to(device)

In [118]:
vocab_size = tokenizer.vocab_size

In [119]:
from transformers import AutoTokenizer

# 1️⃣ Clean your text column
x_test['text'] = x_test['text'].astype(str)        # convert all entries to strings
x_test['text'] = x_test['text'].fillna('')         # replace NaN with empty string

# 2️⃣ Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# 3️⃣ Tokenize safely
encodings2 = tokenizer(
    list(x_test['text']),   # guaranteed list of strings
    padding='max_length',
    truncation=True,
    max_length=33,
    return_tensors='pt'
)

print("input_ids shape:", encodings['input_ids'].shape)


input_ids shape: torch.Size([27481, 33])


In [120]:
x_test = encodings2.get('input_ids').to(device)

In [121]:
y_test = y_test.apply(lambda x: 0 if x == 'neutral' else (1 if x == 'positive' else 2))

In [122]:
y_test = torch.tensor(y_test).to(device)

In [123]:
x_train

tensor([[  101,  1045,  1036,  ...,     0,     0,     0],
        [  101, 17111,  2080,  ...,     0,     0,     0],
        [  101, 18917,  2033,  ...,     0,     0,     0],
        ...,
        [  101,  8038,  2100,  ...,     0,     0,     0],
        [  101,  2021,  2009,  ...,     0,     0,     0],
        [  101,  2035,  2023,  ...,     0,     0,     0]], device='cuda:0')

In [124]:
vocab_size

30522

In [155]:
import torch
from torch.utils.data import DataLoader, TensorDataset

dataset = TensorDataset(x_train, y_train)
dataset
batch_size = 10 # or any batch size you want
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [156]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim)
        self.emb_dropout = nn.Dropout(p=0.56)                       # 🔹 optional: embedding dropout
        self.lstm = nn.LSTM(input_size=embed_dim,
                            hidden_size=hidden_dim,
                            num_layers=num_layers,
                            batch_first=True,
                            dropout=0.53) # LSTM dropout only applied between layers
        self.out_norm = nn.LayerNorm(hidden_dim)                   # 🔹 normalize per time step over H

    def forward(self, x):
        # x: (B, T)
        embeddings = self.embedding(x)             # (B, T, E)
        embeddings = self.emb_dropout(embeddings)  # 🔹 regularize embeddings
        outputs, (h, c) = self.lstm(embeddings)    # outputs: (B, T, H)
        outputs = self.out_norm(outputs)           # 🔹 stabilize before attention
        return outputs, (h, c)

In [ ]:
# class Attention(nn.Module):
#    def __init__(self):
#        super().__init__()
#        self.linear = nn.Linear(in_features= hidden_dim*2, out_features=1)
#        self.softmax = nn.Softmax(dim=2)
#    def forward(self, hidden, enc_outputs):
#        print("Enc_Outputs = ", enc_outputs.shape)
#        print("hidden = ", hidden.shape)
#        hid_shape = hidden.shape[1]
#        enc_shape = enc_outputs.shape[1]
#        hidden = hidden.permute(1, 0, 2)
#        enc_outputs = enc_outputs.repeat(1, hid_shape, 1)
#        hidden = hidden.repeat(1, enc_shape, 1)
#        print("Enc_Outputs = ", enc_outputs.shape)
#        print("hidden = ", hidden.shape)
#        concatenated = torch.concat((hidden, enc_outputs), dim = 2)
#        output = self.linear(concatenated)
#        attn_wgts = self.softmax(output)
#        attn_wgts = attn_wgts.permute(0, 2, 1)
#        print("Attention Weights = ", attn_wgts.shape)
#        print("Enc_Outputs = ", enc_outputs.shape)
#        cosine_prod = torch.bmm(attn_wgts, enc_outputs)
#        return cosine_prod
# class Decoder(nn.Module):
#    def __init__(self):
#        super().__init__()
#        self.embeddings = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim)
#        self.attn = Attention()
#        self.lstm = nn.LSTM(input_size=hidden_dim+embed_dim, num_layers=num_layers, hidden_size=hidden_dim, batch_first= True)
#        self.linear = nn.Linear(in_features=hidden_dim, out_features=vocab_size)
#    def forward(self, y, hidden, dec_outputs):
#        embeddings = self.embeddings(y).unsqueeze(1)
#        attn_wgts = self.attn(hidden[0], enc_outputs)
#        concatenated = torch.concat((attn_wgts, embeddings), dim = 2)
#        lstm_output, h = self.lstm(concatenated.squeeze(2))
#        output = self.linear(lstm_output.squeeze(1))
#        return output, h

In [157]:
class Attention(nn.Module):
    def __init__(self):
        super().__init__()
        self.pre_norm = nn.LayerNorm(hidden_dim)   # 🔹 normalize inputs to attention
        self.linear = nn.Linear(hidden_dim, 1)
        self.softmax = nn.Softmax(dim=1)           # normalize across time steps
        self.dropout = nn.Dropout(p=0.35)
        # 🔹 normalize context vector (optional, but helpful)

    def forward(self, enc_outputs):
        # enc_outputs: (B, T, H)
        enc_outputs = self.pre_norm(enc_outputs)             # 🔹 LN over H at each time step
        scores = self.linear(enc_outputs)                    # (B, T, 1)
        attn_weights = self.softmax(scores)                  # (B, T, 1) sums to 1 over T
        attn_weights = self.dropout(attn_weights)            # 🔹 regularize attention distribution
        context = torch.sum(attn_weights * enc_outputs, 1)   # (B, H) weighted sum over T                  # 🔹 stabilize context going to head
        return context, attn_weights

In [158]:
class SentimentClassification(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.attn = Attention()
        self.layer_norm = nn.LayerNorm(hidden_dim)   # 🔹 normalize LSTM output features
        self.dropout = nn.Dropout(p=0.65)             # 🔹 regularize FC layer
        self.linear = nn.Linear(in_features=hidden_dim, out_features=3)
    def forward(self, x):
        enc_outputs, h = self.encoder(x)

        # Normalize encoder outputs before attention
        enc_outputs = self.layer_norm(enc_outputs)

        result, attn = self.attn(enc_outputs)

        # Apply dropout before classification
        result = self.dropout(result)
        logits = self.linear(result)
        return logits, attn

In [151]:
model = SentimentClassification()
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()


In [159]:
model = model.to(device)

In [160]:
epochs = 10

In [161]:
from sklearn.metrics import precision_recall_fscore_support, classification_report, accuracy_score

for epoch in range(1, epochs + 1):
    model.train()
    epoch_loss = 0.0
    total = 0
    correct = 0

    # for metrics
    all_preds = []
    all_targets = []

    for batch_x, batch_y in dataloader:
        # if you use GPU, move tensors: batch_x = batch_x.to(device); batch_y = batch_y.to(device)

        optimizer.zero_grad()

        logits, attn = model(batch_x)           # logits: (B, C)
        loss = criterion(logits, batch_y)

        loss.backward()
        optimizer.step()

        # bookkeeping
        B = batch_x.size(0)
        epoch_loss += loss.item() * B

        preds = logits.argmax(dim=1)            # (B,)
        correct += (preds == batch_y).sum().item()
        total   += B

        # collect for sklearn
        all_preds.extend(preds.detach().cpu().tolist())
        all_targets.extend(batch_y.detach().cpu().tolist())

    # epoch-level metrics
    avg_loss = epoch_loss / total
    acc = correct / total

    # macro/weighted precision/recall/f1
    p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(
        all_targets, all_preds, average='macro', zero_division=0
    )
    p_weighted, r_weighted, f_weighted, _ = precision_recall_fscore_support(
        all_targets, all_preds, average='weighted', zero_division=0
    )

    print(f"Epoch {epoch:02d} | "
          f"Loss: {avg_loss:.4f} | Acc: {acc:.4f} | "
          f"P_macro: {p_macro:.4f} R_macro: {r_macro:.4f} F_macro: {f_macro:.4f} | "
          f"P_w: {p_weighted:.4f} R_w: {r_weighted:.4f} F_w: {f_weighted:.4f}")

    # (optional) detailed per-class report
    # label_names = ["negative", "neutral", "positive"]  # adjust to your mapping/order
    # print(classification_report(all_targets, all_preds, target_names=label_names, zero_division=0))


Epoch 01 | Loss: 0.8367 | Acc: 0.6024 | P_macro: 0.5425 R_macro: 0.5588 F_macro: 0.5183 | P_w: 0.5600 R_w: 0.6024 F_w: 0.5519
Epoch 02 | Loss: 0.8289 | Acc: 0.6066 | P_macro: 0.5473 R_macro: 0.5635 F_macro: 0.5300 | P_w: 0.5650 R_w: 0.6066 F_w: 0.5626
Epoch 03 | Loss: 0.8216 | Acc: 0.6078 | P_macro: 0.5453 R_macro: 0.5634 F_macro: 0.5214 | P_w: 0.5634 R_w: 0.6078 F_w: 0.5558
Epoch 04 | Loss: 0.8239 | Acc: 0.6093 | P_macro: 0.5471 R_macro: 0.5651 F_macro: 0.5271 | P_w: 0.5656 R_w: 0.6093 F_w: 0.5610
Epoch 05 | Loss: 0.8140 | Acc: 0.6136 | P_macro: 0.5511 R_macro: 0.5693 F_macro: 0.5293 | P_w: 0.5697 R_w: 0.6136 F_w: 0.5636
Epoch 06 | Loss: 0.8098 | Acc: 0.6200 | P_macro: 0.5639 R_macro: 0.5768 F_macro: 0.5441 | P_w: 0.5818 R_w: 0.6200 F_w: 0.5770
Epoch 07 | Loss: 0.8036 | Acc: 0.6197 | P_macro: 0.5616 R_macro: 0.5768 F_macro: 0.5457 | P_w: 0.5800 R_w: 0.6197 F_w: 0.5784
Epoch 08 | Loss: 0.8033 | Acc: 0.6189 | P_macro: 0.5580 R_macro: 0.5760 F_macro: 0.5440 | P_w: 0.5772 R_w: 0.6189 F_w:

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset

dataset = TensorDataset(x_test, y_test)
dataset
batch_size = 2  # or any batch size you want
test_dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [ ]:
correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for batch_x, batch_y in test_dataloader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device

        logits, attn = model(batch_x)
        preds = torch.argmax(logits, dim=1)  # Get predicted class index

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch_y.cpu().numpy())

        correct += (preds == batch_y).sum().item()
        total += batch_y.size(0)

accuracy = correct / total
print(f"Test Accuracy: {accuracy:.4f}")


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(all_labels, all_preds, target_names=["negative", "neutral", "positive"]))


In [ ]:
model.eval() 

In [ ]:
import torch
from transformers import AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()  # disable dropout, etc.

# --- tokenizer ---
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# --- example sentence ---
text = "I will accept it, it happily, and nicely"

# Tokenize -> get only input_ids (your LSTM uses just IDs)
enc = tokenizer(
    text,
    padding='max_length',
    truncation=True,
    max_length=33,          # pick a value your model was trained with
    return_tensors='pt'
)

input_ids = enc["input_ids"].to(device)   # (1, T)

with torch.no_grad():
    logits, attn = model(input_ids)       # logits: (1, 3), attn: (1, T, 1)

probs = torch.softmax(logits, dim=-1).squeeze(0)   # (3,)
pred_id = torch.argmax(probs).item()

id2label = {0: "neutral", 1: "positive", 2: "negative"}  # <- use your mapping
print("Prediction:", id2label[pred_id])
print("Probs:", {id2label[i]: float(probs[i]) for i in range(3)})


In [ ]:
import torch

# Save only model parameters
torch.save(model.state_dict(), "sentiment_model_latest_with_200_epochs.pth")
print("✅ Model weights saved successfully!")


In [41]:
import tensorflow as tf

2025-10-21 11:14:54.560527: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761045294.746408     150 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761045294.804339     150 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [42]:
from tensorflow.keras import layers, Model

In [ ]:
%config IPCompleter.use_jedi = True
%config Completer.use_jedi = True

In [ ]:
%config IPCompleter.greedy = True

In [59]:
x_train = x_train.cpu().numpy()
y_train = y_train.cpu().numpy()

In [43]:
import tensorflow as tf

In [44]:
from tensorflow.keras import layers

In [50]:
class Encoder(tf.keras.Model):
      def __init__(self):
          super().__init__()
          self.embeddings = layers.Embedding(input_dim = vocab_size, output_dim = embed_dim)
          self.lstm = layers.LSTM(units = embed_dim, activation='tanh', return_sequences=True)
      def call(self, x):
          embeddings = self.embeddings(x)
          output = self.lstm(embeddings)
          return output

In [51]:
class Attention(tf.keras.Model):
      def __init__(self):
          super().__init__()
          self.Linear = layers.Dense(units = 1)
          self.dropout = layers.Dropout(0.56)
      def call(self, enc_outputs):
          output = self.Linear(enc_outputs)
          output = self.dropout(output)
          attn_weights = tf.nn.softmax(output, axis=1)
          attented_output = tf.reduce_sum(attn_weights * enc_outputs, axis = 1)
          return attented_output

In [52]:
class SentimentClassification(tf.keras.Model):
      def __init__(self):
          super().__init__()
          self.encoder = Encoder()
          self.attn = Attention()
          self.linear = layers.Dense(units=3, activation = 'softmax')
          self.dropout = layers.Dropout(0.35)
      def call(self, x): 
          enc_outputs = self.encoder(x)
          attn_scores = self.attn(enc_outputs)
          final_output = self.linear(attn_scores)
          #print("Final Output = ", final_output.shape)
          return final_output

In [53]:
batch_size = 10

def simple_generator(x, y, batch_size=10):
    for i in range(0, len(x), batch_size):
        yield x[i:i + batch_size], y[i:i + batch_size]


In [54]:
train_loss = tf.keras.metrics.Mean(name='train_loss')
train_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')

In [55]:
def train_step(x_batch, y_batch):
    """Single training step with gradient computation"""
    with tf.GradientTape() as tape:
        predictions = model(x_batch, training=True)
        #print("Predictions = ", predictions)
        loss = loss_fn(y_batch, predictions)
    
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    
    train_loss(loss)
    train_accuracy(y_batch, predictions)
    
    return loss

In [57]:
epochs = 10
model = SentimentClassification()

optimizer = tf.keras.optimizers.Adam(1e-3)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)

In [60]:
for epoch in range(epochs):
    # Reset metrics
    train_loss.reset_state()
    train_accuracy.reset_state()
    
    # Training loop
    for x_batch, y_batch in simple_generator(x_train, y_train):
        # Convert to TensorFlow tensors
        x_batch_tf = tf.convert_to_tensor(x_batch, dtype=tf.int32)
        y_batch_tf = tf.convert_to_tensor(y_batch, dtype=tf.int32)
        
        train_step(x_batch_tf, y_batch_tf)
    
    # Print metrics
    print(f"Epoch {epoch+1}/{epochs} | "
          f"Loss: {train_loss.result():.4f} | "
          f"Accuracy: {train_accuracy.result():.4f}")

I0000 00:00:1761045442.019966     192 cuda_dnn.cc:529] Loaded cuDNN version 90300


Epoch 1/10 | Loss: 0.5796 | Accuracy: 0.7657
Epoch 2/10 | Loss: 0.3613 | Accuracy: 0.8682
Epoch 3/10 | Loss: 0.2857 | Accuracy: 0.8999
Epoch 4/10 | Loss: 0.2342 | Accuracy: 0.9198
Epoch 5/10 | Loss: 0.1965 | Accuracy: 0.9347
Epoch 6/10 | Loss: 0.1636 | Accuracy: 0.9466
Epoch 7/10 | Loss: 0.1381 | Accuracy: 0.9552
Epoch 8/10 | Loss: 0.1222 | Accuracy: 0.9597
Epoch 9/10 | Loss: 0.1061 | Accuracy: 0.9653
Epoch 10/10 | Loss: 0.0953 | Accuracy: 0.9686


In [62]:
x_test = x_test.cpu().numpy()
y_test = y_test.cpu().numpy()

In [ ]:
x_test

In [63]:
# Define test metrics
test_loss = tf.keras.metrics.Mean(name="test_loss")
test_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name="test_accuracy")

@tf.function
def test_step(x_batch, y_batch):
    predictions = model(x_batch, training=False)
    loss_value = loss_fn(y_batch, predictions)

    test_loss.update_state(loss_value)
    test_accuracy.update_state(y_batch, predictions)

# --- EVALUATE ---
test_loss.reset_state()
test_accuracy.reset_state()

for x_batch, y_batch in simple_generator(x_test, y_test):
    x_batch_tf = tf.convert_to_tensor(x_batch, dtype=tf.int32)
    y_batch_tf = tf.convert_to_tensor(y_batch, dtype=tf.int32)
    test_step(x_batch_tf, y_batch_tf)

print(f"Test Loss: {test_loss.result():.4f} | Test Accuracy: {test_accuracy.result():.4f}")


Test Loss: 6.8529 | Test Accuracy: 0.1358


In [85]:
from transformers import AutoTokenizer
import numpy as np

# Load the same tokenizer used during training
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Example sentence
text = "It's delicious"

# Tokenize & pad
inputs = tokenizer(
    text,
    padding='max_length',
    truncation=True,
    max_length=33,
    return_tensors='tf'
)

# Predict
predictions = model(inputs['input_ids'], training=False)
predictions = predictions.numpy()[0]
for i, vals in enumerate(predictions):
      print(f"{i}: {vals}")
predictions = model(inputs['input_ids'], training=False)
pred_class = tf.argmax(predictions, axis=1).numpy()[0]
#print(f"{vals.get()}: {}") for vals in predictions
# Map label to sentiment
label_map = {2: "negative", 0: "neutral", 1: "positive"}
print(f"Sentence: {text}")
print(f"Predicted Sentiment: {label_map[pred_class]}")
#print(f"Other probabilites:- {}") for key, values  

0: 0.0038717694114893675
1: 0.9535945057868958
2: 0.04253377392888069
Sentence: It's delicious
Predicted Sentiment: positive


In [71]:
model.save("sentiment_model.keras")

In [ ]:
for epoch in range(epochs):
    epoch_loss = 0
    for x_batch, y_batch in simple_generator(x_train, y_train, batch_size):
        with tf.GradientTape() as tape:
            preds = model(x_batch, training=True)
            loss = loss_fn(y_batch, preds)

        grads = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        epoch_loss += loss.numpy()

    print(f"Epoch {epoch+1}, Loss: {epoch_loss:.4f}")
